In [9]:
# ==============================
# IMPORTS
# ==============================
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ==============================
# AUTO DATASET DETECTION
# ==============================
base_path = "/kaggle/input"
datasets = os.listdir(base_path)

print("Available datasets:", datasets)

DATASET_PATH = os.path.join(base_path, datasets[0])
print("Using dataset:", DATASET_PATH)

folders = os.listdir(DATASET_PATH)

train_path, test_path = None, None
for f in folders:
    if "train" in f.lower():
        train_path = os.path.join(DATASET_PATH, f)
    if "test" in f.lower():
        test_path = os.path.join(DATASET_PATH, f)

print("Train:", train_path)
print("Test:", test_path)

# ==============================
# DATA PREPROCESSING
# ==============================
train_datagen = ImageDataGenerator(rescale=1./255, zoom_range=0.2, horizontal_flip=True)
test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_path, target_size=(224,224), batch_size=32, class_mode='categorical'
)

test_data = test_datagen.flow_from_directory(
    test_path, target_size=(224,224), batch_size=32, class_mode='categorical'
)

# ==============================
# CNN MODEL
# ==============================
model = Sequential()

model.add(Conv2D(32,(3,3),activation='relu',input_shape=(224,224,3), name="conv1"))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64,(3,3),activation='relu', name="conv2"))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(128,(3,3),activation='relu', name="conv3"))
model.add(MaxPooling2D(2,2))

model.add(Flatten())
model.add(Dense(128,activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(train_data.num_classes, activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

# ==============================
# TRAIN
# ==============================
history = model.fit(train_data, epochs=5, validation_data=test_data)

# ==============================
# FILTER VISUALIZATION
# ==============================
layer_outputs = [layer.output for layer in model.layers if 'conv' in layer.name]
activation_model = Model(inputs=model.input, outputs=layer_outputs)

sample_img = next(test_data)[0][:1]
activations = activation_model.predict(sample_img)

print("\n--- Filter Visualization ---")
for act in activations:
    plt.figure(figsize=(10,4))
    for i in range(min(6, act.shape[-1])):
        plt.subplot(1,6,i+1)
        plt.imshow(act[0,:,:,i], cmap='viridis')
        plt.axis('off')
    plt.show()

# ==============================
# GRAD-CAM
# ==============================
def gradcam(model, img_array, layer_name="conv3"):
    grad_model = Model(
        [model.inputs],
        [model.get_layer(layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_output, predictions = grad_model(img_array)
        loss = predictions[:, np.argmax(predictions[0])]

    grads = tape.gradient(loss, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))

    conv_output = conv_output[0]
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = np.maximum(heatmap, 0) / np.max(heatmap)
    return heatmap.numpy()

# Apply Grad-CAM
img = sample_img
heatmap = gradcam(model, img)

# Overlay heatmap
img_display = img[0]
heatmap = cv2.resize(heatmap, (224,224))
heatmap = np.uint8(255 * heatmap)

heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
superimposed = heatmap * 0.4 + img_display * 255

plt.imshow(superimposed.astype('uint8'))
plt.title("Grad-CAM")
plt.axis('off')
plt.show()

# ==============================
# LOCALIZATION QUALITY (DICE)
# ==============================
def dice_score(pred, true):
    intersection = np.sum(pred * true)
    return (2. * intersection) / (np.sum(pred) + np.sum(true) + 1e-8)

# Dummy example (since no mask dataset)
pred_mask = (heatmap > 100).astype(int)
true_mask = (heatmap > 120).astype(int)

dice = dice_score(pred_mask, true_mask)
print("Dice Score:", dice)

# ==============================
# MEDICAL INTERPRETATION
# ==============================
class_names = list(train_data.class_indices.keys())
prediction = model.predict(sample_img)

pred_class = class_names[np.argmax(prediction)]
confidence = np.max(prediction)

print("\n--- Medical Interpretation ---")
print(f"Predicted Condition: {pred_class}")
print(f"Confidence: {confidence:.2f}")

if pred_class != "no_tumor":
    print("⚠ Tumor detected. Recommend clinical evaluation.")
else:
    print("✅ No tumor detected.")